In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns


In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

def datetime_to_decimal_year(datetime_):
    year = datetime_.year
    days = (datetime_.date() - datetime.date(year, 1,1)).days
    res = year + days/365.25
    return res

def FFT_res(x, si):
    n = len(x)
    x_FFT_shifted = np.fft.fftshift(np.fft.fft(x))
    x_FFT_freq_shifted = np.fft.fftshift(np.fft.fftfreq(n, d=si))
    return x_FFT_freq_shifted, x_FFT_shifted


def fft_noramlize(fft_res):
    n = len(fft_res)
    norm_fft = np.abs(fft_res)/(np.max(np.abs(fft_res)))
    return norm_fft

def line_fit(X,Y):
    P = np.polyfit(X,Y,1)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.05, np.max(X)+dx*0.05])
    Y_new = np.polyval(P, X_new)
    return X_new, Y_new, P

In [ ]:
window_size = 3
window_size2 = 36


df = pd.read_excel('interp_20251027_.xlsx')
paths = list(df['path_name'].value_counts().index.values)
for path in paths:
    df.loc[df['path_name']==path, "path_ma"] = df.loc[df['path_name']==path, 'd13c'].rolling(window_size, center=True).mean()
#df[['d13c_ma','d18o_ma']] = df[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted = df.sort_values("age_absolute")
df_sorted['age_ws'] = df_sorted['age_absolute'].rolling(window_size, center=True).mean()
df_sorted['age_ws2'] = df_sorted['age_absolute'].rolling(window_size2, center=True).mean()

df_sorted[['d13c_ma','d18o_ma']] = df_sorted[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted[['d13c_ma2','d18o_ma2']] = df_sorted[['d13c','d18o']].rolling(window_size2, center=True).mean()
df_sorted[['d13c_std','d18o_std']] = df_sorted[['d13c','d18o']].rolling(window_size2, center=True).std()
df_sorted['d13c_no_tectonic'] = df_sorted['d13c'] - df_sorted['d13c_ma2'].fillna(0)
df_sorted['d18o_no_tectonic'] = df_sorted['d18o'] - df_sorted['d18o_ma2'].fillna(0)
df_sorted[['d13c_notec_ma','d18o_notec_ma']] = df_sorted[['d13c_no_tectonic','d18o_no_tectonic']].rolling(window_size, center=True).mean()

#-----------------------RESAMPLE MEAN & STD-----------------------------------------------------

df_sorted['age_dt'] = decimal_year_to_datetime(df_sorted['age_absolute'])
df_resampled_ = df_sorted.set_index('age_dt')
df_resampled_.drop(['Name', 'Mass', 'Amp 44', 'Area 44', 'index','sample','age1','path_name',
       'path_num', 'path_ma', 'age_ws', 'age_ws2',
       'd13c_ma', 'd18o_ma', 'd13c_ma2', 'd18o_ma2', 'd13c_std', 'd18o_std'], axis=1, inplace=True)
df_resampled = df_resampled_.resample('30D').mean()

df_resampled.interpolate(inplace=True)
df_resampled[['d13c_ma','d18o_ma']] = df_resampled[['d13c','d18o']].rolling(window_size, center=True).mean()
df_resampled[['d13c_ma2','d18o_ma2']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).mean()
df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window_size2, center=True).std()

#-----------------------------------------------------------------------------------------------


#-----------------------CENTRAL DIFF METHOD-----------------------------------------------------
window = 12
col = 'd18o'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])

window = 12
col = 'd13c'

last = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[-1])
first = df_resampled.rolling(window, center=True).agg(lambda x: x.iloc[0])
df_resampled[col+'_cdm'] = (last[col] - first[col]) / (last['age_absolute'] - first['age_absolute'])
#-----------------------------------------------------------------------------------------------


#-----------------------------Moving OSL---------------------------------------
coefs_centered = pd.DataFrame()

# Define multiple windows and target columns
windows = [36, 36]
cols = ['d18o', 'd13c']

for window, col in zip(windows, cols):
    
    # Prepare data (assumes df_resampled contains all variables)
    data = df_resampled[[col, 'age_absolute']].dropna()

    if len(data) < window:
        print(f"Not enough data for column '{col}' with window {window}. Skipping.")
        continue
    
    # Setup endogenous and exogenous variables
    endog = data[col]
    exog = sm.add_constant(data['age_absolute'])

    # Fit rolling OLS
    rols = RollingOLS(endog, exog, window=window)
    rres = rols.fit()

    # Shift result to center and rename columns
    coefs = rres.params.shift(-(window // 2)).dropna()
    coefs = coefs.rename(columns={'const': f'{col}_intercept', 'age_absolute': f'{col}_slope'})

    # Merge into result
    coefs_centered = coefs_centered.join(coefs, how='outer')

# Add shared age column (assumes time index is same)
coefs_centered['age_absolute'] = df_resampled['age_absolute']

# Drop rows with NaNs in all slope columns (optional)
slope_cols = [f"{col}_slope" for col in cols]
coefs_centered.dropna(subset=slope_cols, how='all', inplace=True)
 
#-----------------------------------------------------------------------------------------------


#---------------------------------------FFT-----------------------------------------------------
d13c_fft_freq, d13c_fft = FFT_res(df_resampled['d13c'], 30)
d18o_fft_freq, d18o_fft = FFT_res(df_resampled['d18o'], 30)

df_resampled[['d13c_std','d18o_std']] = df_resampled[['d13c','d18o']].rolling(window=window_size2, center=True).std()
d13c_stds = df_resampled.dropna()['d13c_std']
d18o_stds = df_resampled.dropna()['d18o_std']
d13c_std_fft_freq, d13c_std_fft = FFT_res(d13c_stds, 30)
d18o_std_fft_freq, d18o_std_fft = FFT_res(d18o_stds, 30)
#-----------------------------------------------------------------------------------------------


df_resampled

In [ ]:
fred = pd.read_excel('../2025_08_22_FredsData-post2008/FredData_editedMemo.xlsx').dropna()
fred['d18o_pdb'] = -(fred['d18o']-20.91)/1.03091
fred[['d13c_ma','d18o_ma']] = fred[['d13c','d18o_pdb']].rolling(window_size, center=True).mean()
fred[['d13c_std','d18o_std']] = fred[['d13c','d18o_pdb']].rolling(10, center=True).std()
fred_resampled = fred.drop(fred.columns[3:], axis=1)
fred_resampled['age_dt'] = decimal_year_to_datetime(fred_resampled['decimal_year'])
fred_resampled.set_index('age_dt', inplace=True)
fred_resampled = fred_resampled.resample('30D').mean()
fred_resampled.interpolate(inplace=True)
fred_resampled

In [ ]:
tide = pd.read_csv('../2025_10_10_Fred-TideGauge/memo_edited.csv')
tide['age_dt'] = decimal_year_to_datetime(tide['Date'])#.dt.date
tide = tide.set_index('age_dt')


In [ ]:
enso_phase = pd.read_csv('../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')


enso = pd.read_csv("../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']


margin = 0.5
enso['phase'] = 'N'
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso['phase_group'] = (enso['phase'] != enso['phase'].shift()).cumsum()
blocks = enso.groupby('phase_group').agg(
    start=('Date', 'first'),
    end=('Date', 'last'),
    value=('phase', 'first')
).reset_index(drop=True)


enso_fft_freq, enso_fft = FFT_res(enso['ENSO'], 30)


enso


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  text=df_sorted['Name'],  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c_no_tectonic'],  mode="markers", marker_color="lightgreen",  name='d13C without tectonic', text=df_sorted['Name']),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_notec_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)



#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)
fig.add_trace(go.Scatter(x=tide['Date'], y=tide['Sea Level'], mode="lines", line_color='#120C45', name='Tide Gauge'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=False)



fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)





#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

#fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
#                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
#                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

#fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
#                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
#                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

#fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
#        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Tide Gauge [mm]", secondary_y=True)

fig.show()
#fig.write_html('d13c_tidegauge.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o'],  mode="markers", marker_color="lightblue",  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma'], mode="lines", line_color='navy', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma2'], mode="lines", line_color='orange', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_pdb'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_std'], mode="lines", line_color='darkred', name='std'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_std'], mode="lines", line_color='darkred', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.85], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='N Gizo EQ 1936 Apr'))

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[25.66], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE


fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[25.6], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1992.5, y=25.7, text="?", showarrow=False, font=dict(size=35, color="red"))

fig.add_annotation(x=1991.4, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=20, ay=-50)

fig.add_annotation(x=1974.3, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-20, ay=40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d18o", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta O ^{18}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d18o_std.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o'],  mode="markers", marker_color="lightblue", text=df_sorted['Name'], name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma'], mode="lines", line_color='navy', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_ma2'], mode="lines", line_color='orange', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_pdb'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d18o_std'], mode="lines", line_color='darkred', name='std'), secondary_y=True)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d18o_std'], mode="lines", line_color='darkred', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)


#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

for i, index in enumerate(blocks.index):
    block = blocks.loc[index]
    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
                  x1=datetime_to_decimal_year(block['end']), 
                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[25.85], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='N Gizo EQ 1936 Apr'))

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[25.66], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE


fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[25.6], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

#fig.add_annotation(x=1992.5, y=25.7, text="?", showarrow=False, font=dict(size=35, color="red"))

fig.add_annotation(x=1991.4, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=20, ay=-50)

fig.add_annotation(x=1974.3, y=25.6, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-20, ay=40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d18o", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= r'$\delta O ^{18}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d18o_std.html')


In [ ]:
fig, axes = plt.subplots(3, figsize=(10,7), sharex=True,height_ratios=[3,1,1], gridspec_kw={'hspace':0})

ax = axes[0]

ax.plot(df_resampled['age_absolute'], df_resampled['d18o'], ".", color='lightblue', alpha=0.7, label=r'$\delta^{18}O$' )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma'],  color='navy', lw=0.75 )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d18o_pdb'], ".", color='#40E0D0', alpha=0.3,  )
ax.plot(fred['decimal_year'], fred['d18o_ma'],  color='#1AC4A8', lw=0.75 )


ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(25, 26.57)

ax.grid()
ax.legend(loc='lower left')



ax = axes[1]

ax.plot(df_resampled['age_absolute'], df_resampled['d18o_std'], color='darkred' , label='std d18O'  )
ax.plot(fred['decimal_year'], fred['d18o_std'], '--', color='darkred' , label='fred std d18O'  )

ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=-1, ymax=1, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(-0.05,0.35)
ax.grid()
ax.legend(loc='lower left')



ax = axes[2]

ax.plot(coefs_centered['age_absolute'], coefs_centered['d13c_slope'], color='pink' , label='moving OLS'  )

ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
          ymin=-1, ymax=1, color=['#F54927','#F5C827','#2D8708'])

ax.set_ylim(-0.5,0.5)
ax.grid()
ax.legend(loc='lower left')

#fig.savefig('d18O_std_OLS.pdf')

In [ ]:
phase_color = {'W':'red', 'C':'blue','N':'white'}
#enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2014)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C', text=df_sorted['Name']),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma2'], mode="lines", line_color='red', name='average'), secondary_y=False)



fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c'], mode="markers", line_color='turquoise', name='fred'), secondary_y=False)
fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_ma'], mode="lines", line_color='darkturquoise', name='average'), secondary_y=False)


#fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_std'], mode="lines", line_color='darkblue', name='std'), secondary_y=True)
#fig.add_trace(go.Scatter(x=fred['decimal_year'], y=fred['d13c_std'], mode="lines", line_color='darkblue', name='std_fred'), secondary_y=True)

#fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

#fig.add_trace(go.Scatter(x=coefs_centered['age_absolute'], y=coefs_centered['d13c_slope'], mode="lines", line_color='pink', name='derivative'), secondary_y=True)


fig.update_layout(
    yaxis2=dict(title="Secondary Y-axis Title", overlaying="y", side="right", gridcolor='lightgray', showgrid=True)) # Grid for secondary Y-axis)

#fig.add_vline(x=1936+(31+28+22)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1936 Mar')
#fig.add_vline(x=1936+(31+28+31+19)/365, line_width=2, line_dash="dash", line_color="red", opacity=1, showlegend=True,name='N Gizo EQ 1936 Apr')
#fig.add_vline(x=1942+(12)/365, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='W Gizo EQ 1942 Jan')
#fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
#fig.add_vline(x=1969+159/365, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Vangunu Reef 1969 EQ M5.9')
#fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="yellow", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
#fig.add_vline(x=2007.25, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

#for i, index in enumerate(blocks.index):
#    block = blocks.loc[index]
#    fig.add_vrect(x0=datetime_to_decimal_year(block['start']), 
#                  x1=datetime_to_decimal_year(block['end']), 
#                  fillcolor=phase_color[block['value']], opacity=0.1)

fig.add_trace(go.Scatter(x=[1936+(31+29+22)/365], y=[-0.975], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='red', width=2)),
                        name='M7.1 EQ 1936 Apr')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=903573&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[1953-(25)/365], y=[-1.395], mode='markers', 
                         marker=dict(size=15, symbol='star', color='yellow', line=dict(color='purple', width=2)),
                        name='M7.0 EQ 1952 Dec')) #https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=893860&out_format=ISF2&request=COMPREHENSIVE

fig.add_trace(go.Scatter(x=[2007+(31+29+31)/365], y=[-2.183], mode='markers', 
                         marker=dict(size=15, symbol='star', color='blue', line=dict(color='orange', width=2)),
                        name='M8.1 EQ 2007 Apr'))

fig.add_annotation(x=1973.9, y=-1.388, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=-40, ay=40)

fig.add_annotation(x=1990.85, y=-1.51, text="?", showarrow=True,  font=dict(size=35, color="red"), 
        align="center", arrowhead=2, arrowsize=1, arrowwidth=2, arrowcolor="red", ax=30, ay=-40)


# Add figure title
fig.update_layout(width=1200, height=600, title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="Standard Deviation", secondary_y=True)

fig.show()
#fig.write_html('d13c_std.html')


In [ ]:
fig, axes = plt.subplots(3, figsize=(6,4), sharex=True,gridspec_kw={'hspace':0} )

for i in range(3):
    ax = axes[i]
    if i==0:
        ax.plot(1/d18o_fft_freq[1:], fft_noramlize(d18o_fft[1:])**2,label='d18O')
    if i==1:
        ax.plot(1/d13c_fft_freq[1:], fft_noramlize(d13c_fft[1:])**2, color='green',label='d13C')
    if i==2:
        ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:])**3, color='gray', label='enso')
    ax.legend(loc='center left')  
    #days_freq = 271
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
    #days_freq = 281
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='365 days')



    ax.set_xlim(200,400)
#ax.set_xlim(200,3600)
#ax.set_xscale('log')
    ax.set_ylim(0,0.003)
    plt.yticks([0,0.001, 0.002])
axes[0].set_ylim(0,0.000001)
#plt.yticks([0.5e-6, 1e-6])
ax.set_xlabel('Period [days]')
axes[1].set_ylabel('Normalized FFT Amplitude')
axes[0].set_title('Frequency Spectrum')
axes[1].legend(loc='center left')

#ax.set_title('Moving Standard Deviation FFT')
plt.show()
#fig.savefig('std_fft.pdf')



In [ ]:
fig, axes = plt.subplots(3, figsize=(6,4), sharex=True,gridspec_kw={'hspace':0} )

for i in range(3):
    ax = axes[i]
    if i==0:
        ax.plot(1/d18o_std_fft_freq[1:], fft_noramlize(d18o_std_fft[1:]),label=r'$\delta^{18}O$')
    if i==1:
        ax.plot(1/d13c_std_fft_freq[1:], fft_noramlize(d13c_std_fft[1:]), color='green',label=r'$\delta^{13}C$')
    if i==2:
        ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:]), color='gray', label='enso')
    
    ax.legend(loc='upper left')  
    days_freq = 271
    ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
    days_freq = 281
    ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='1 Y')

    days_freq = 746    
    ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    
    
    ax.set_xlim(100,300)
    #ax.set_xlim(200,3600)
    #ax.set_xscale('log')
    #plt.yticks([0,0.001])
    ax.set_ylim(0,0.005)

axes[2].set_ylim(0,0.072)
ax.set_xlabel('Period [days]')
axes[1].set_ylabel('Normalized FFT Amplitude')
axes[0].set_title('Frequency Spectrum ')
axes[1].legend(loc='upper left')  
#ax.legend(loc=[500,0])
#ax.set_title('Moving Standard Deviation FFT')
plt.show()
#fig.savefig('std_fft.pdf')


In [ ]:
# Ensure df_resampled is sorted by date
df = df_resampled[['d13c','d18o', 'd13c_no_tectonic', 'd18o_no_tectonic', 'd13c_notec_ma']].copy()
df = df.sort_index()

# Target dates to align with
target_dates = pd.to_datetime(enso.loc[:685, 'Date'])

# Interpolate to get values at those dates
aligned = df.reindex(df.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned['age_absolute'] = np.array([datetime_to_decimal_year(x) for x in pd.to_datetime(aligned.index)])
aligned['ENSO'] = enso.loc[:685, 'ENSO'].values
aligned_tidegauge = tide.reindex(tide.index.union(target_dates)).interpolate(method='time').loc[target_dates]
aligned = aligned.join(aligned_tidegauge['Sea Level'].dropna())
aligned.loc['2005-01-01':'2008-01-01']

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_resampled['age_absolute'], y=df_resampled['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)
fig.add_trace(go.Scatter(x=aligned['age_absolute'], y=aligned['d13c'], mode="markers", line_color='red', name='enso_aligned'), secondary_y=False)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="markers", line_color='gray', name='ENSO'), secondary_y=True)


In [ ]:
correlation_matrix = aligned.loc['1969-01-01':'1975-01-01'].corr()

print(correlation_matrix)

In [ ]:
correlation_matrix = aligned.corr()

# Create a heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt=".2f")
plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.show()


In [ ]:
#---------------------------------------FFT-----------------------------------------------------
d13c_aligned_fft_freq, d13c_aligned_fft = FFT_res(aligned['d13c'], 30)
d18o_aligned_fft_freq, d18o_aligned_fft = FFT_res(aligned['d18o'], 30)
#-----------------------------------------------------------------------------------------------

In [ ]:
fig, axes = plt.subplots(3, figsize=(6,4), sharex=True,gridspec_kw={'hspace':0} )

for i in range(3):
    ax = axes[i]
    if i==0:
        ax.plot(1/d18o_aligned_fft_freq[1:], fft_noramlize(d18o_aligned_fft[1:]),label=r'$\delta^{18}O$')
    if i==1:
        ax.plot(1/d13c_aligned_fft_freq[1:], fft_noramlize(d13c_aligned_fft[1:]), color='green',label=r'$\delta^{13}C$')
    if i==2:
        ax.plot(1/enso_fft_freq[1:], fft_noramlize(enso_fft[1:]), color='gray', label='enso')
    
    ax.legend(loc='upper right')  
    days_freq = 271
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='purple', linestyles='dotted', alpha=0.4, label='{:} days'.format(days_freq))
    days_freq = 281
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    days_freq = 746    
    #ax.vlines(days_freq, ymin=-100, ymax=2500, color='blue', linestyles='dashed', alpha=0.4, label='{:} days'.format(days_freq))
    
    ax.vlines(365, ymin=-100, ymax=2500, color='red', linestyles='solid', alpha=0.4, label='1 Y')
    
    ax.set_xlim(300,700)
    #ax.set_xlim(200,3600)
    #ax.set_xscale('log')
    #plt.yticks([0,0.001])
    
    
axes[0].set_ylim(0,0.001)
axes[1].set_ylim(0,0.04)
axes[2].set_ylim(0,0.4)
ax.set_xlabel('Period [days]')
axes[1].set_ylabel('Normalized FFT Amplitude')
axes[0].set_title('Frequency Spectrum ')
axes[1].legend('upper right')
#ax.set_title('Moving Standard Deviation FFT')
plt.show()
#fig.savefig('std_fft.pdf')


In [ ]:
fig, ax = plt.subplots(1, figsize=(10,5))

ax.plot(df_resampled['age_absolute'], df_resampled['d18o'], ".", color='lightblue', alpha=0.7, label=r'$\delta^{18}O$' )
ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma'],  color='navy', lw=0.75 )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d18o_pdb'], ".", color='#40E0D0', alpha=0.3,  )
ax.plot(fred['decimal_year'], fred['d18o_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

plt.axvline(1936+(31+29+22)/365, color='#F54927', label='M7.1 EQ 1936 Apr')

x, y, p= line_fit(df_resampled.loc[:'1936-04-19','age_absolute'].values, 
               df_resampled.loc[:'1936-04-19','d18o'].values)
ax.plot(x, y, '--', color='#FF9724', linewidth=2.5, label=f'n = {p[0]:.3f}'+r' $\delta^{18}O$ per year')


x, y, p= line_fit(df_resampled.loc['1936-04-19':'1944-12','age_absolute'].values, 
               df_resampled.loc['1936-04-19':'1944-12','d18o'].values)
ax.plot(x, y, '--', color='#90E84D', linewidth=2.5, label=f'n = {p[0]:.3f}'+r' $\delta^{18}O$ per year')


ax.set_ylim(25, 26.57)
ax.set_xlim(1927, 1945)

ax.grid()
ax.legend(loc='lower left')



#fig.savefig('d18O_1936_closeup_trends.pdf')

In [ ]:
fig, ax = plt.subplots(1, figsize=(12,6))

ax.plot(df_resampled['age_absolute'], df_resampled['d13c'], ".", color='lightgreen', alpha=0.7, label=r'$\delta^{13}C$' )
ax.plot(df_resampled['age_absolute'], df_resampled['d13c_ma2'],  color='darkgreen', lw=2.5 )
#ax.plot(df_resampled['age_absolute'], df_resampled['d18o_ma2'],  color='orange', lw=2.5 )


ax.plot(fred['decimal_year'], fred['d13c'], ".", color='#40E0D0', alpha=0.3,  )
ax.plot(fred['decimal_year'], fred['d13c_ma'],  color='#1AC4A8', lw=0.75 )


#ax.vlines([1936+(31+29+22)/365, 1953-(25)/365, 2007+(31+29+31)/365],
#          ymin=20, ymax=30, color=['#F54927','#F5C827','#2D8708'])

plt.axvline(1936+(31+29+22)/365, color='#F54927', label='M7.1 EQ 1936 Apr')
plt.axvline(2007+(31+29+31)/365, color='#F54927', label='M8.1 EQ 2007 Apr')



date1 = '1936-04-19'
date2 = '1944-06'
date2a = '1949-06'
date3 = '1966-02'
date4 = '1978-06'
date5 = '1987-06'
date6 = '2007-01'


                   ###########  1
x, y, p= line_fit(df_resampled.loc[:date1,'age_absolute'].values, 
               df_resampled.loc[:date1,'d13c'].values)
ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  2
x, y, p= line_fit(df_resampled.loc[date1:date2,'age_absolute'].values, 
               df_resampled.loc[date1:date2,'d13c'].values)
ax.plot(x, y, '-', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

x, y, p= line_fit(df_resampled.loc[date2:date2a,'age_absolute'].values, 
               df_resampled.loc[date2:date2a,'d13c'].values)
ax.plot(x, y, '-', color='#FF9724', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  3
x, y, p= line_fit(df_resampled.loc[date2a:date3,'age_absolute'].values, 
               df_resampled.loc[date2a:date3,'d13c'].values)
ax.plot(x, y, '-', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  4
x, y, p= line_fit(df_resampled.loc[date3:date4,'age_absolute'].values, 
               df_resampled.loc[date3:date4,'d13c'].values)
ax.plot(x, y, '--', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  5
x, y, p= line_fit(df_resampled.loc[date4:date5,'age_absolute'].values, 
               df_resampled.loc[date4:date5,'d13c'].values)
ax.plot(x, y, '--', color='#CF1352', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')

                   ###########  6
x, y, p= line_fit(df_resampled.loc[date5:date6,'age_absolute'].values, 
               df_resampled.loc[date5:date6,'d13c'].values)
ax.plot(x, y, linestyle='dotted', color='#1316CF', linewidth=2.5, label=f'{p[0]:.3f}'+r' $\delta^{18}O$ ‰ per year')



ax.set_ylim(-3, 0.5)
ax.set_xlim(1927, 2012)

ax.grid()
#ax.legend(loc='lower left')
ax.legend(bbox_to_anchor=(1, 1))



#fig.savefig('d13C_trens.pdf')

In [ ]:
df_iter =  df_resampled.loc[:date1]

fig, ax = plt.subplots(1 ,figsize=(8, 5))


ax.plot(df_iter['d18o'], df_iter['d13c'], '.')

x, y, p= line_fit(df_iter['d18o'].values, df_iter['d13c'].values)

ax.plot(x, y, '--', color='#90E84D', linewidth=2.5, label=f'n = {p[0]:.3f}')
ax.set_xlabel(r'$\delta^{18}O$')
ax.set_ylabel(r'$\delta^{13}C$')

ax.legend()

In [ ]:
fred['sea_level'].mean()

In [ ]:
fred['sea_level'] = fred['d13c']*37.686-45.662
fred['sea_level_norm'] =  fred['sea_level']-fred['sea_level'].mean()
fred['sea_level_norm']

In [ ]:
tide['Sea Level']

In [ ]:
fig, ax = plt.subplots(1 ,figsize=(8, 5))


ax.plot(fred['decimal_year'] , fred['sea_level_norm'], label='d13c')
ax.plot(tide['Date'], (tide['Sea Level']-tide['Sea Level'].mean())/10, label='tide gauge')
ax.set_xlim(1994,2012)
ax.legend()

In [ ]:
tide_test = pd.read_excel('../2025_10_10_Fred-TideGauge/LUAREN_Honiara_TideGauge.xlsx')
tide_test

In [ ]:
#https://uhslc.soest.hawaii.edu/stations/?stn=009#datums
tide_hawai = pd.read_csv('../2025_10_10_Fred-TideGauge/Hawaii_Data.csv', header=None)
tide_hawai.rename(columns={0:'year', 1:'month', 2:'day', 3:'raw', 4:'ref_corr'}, inplace=True)
tide_hawai['date'] = pd.to_datetime(tide_hawai[['year', 'month', 'day']])
tide_hawai['decimal_year'] = [datetime_to_decimal_year(x) for x in tide_hawai['date'] ]
tide_hawai['raw'] = tide_hawai['raw'].replace(-32767,np.nan)
tide_hawai['ref_corr'] = tide_hawai['raw'] - 707
tide_hawai['date'] = tide_hawai['date'].dt.date
tide_hawai

In [ ]:
fig, ax = plt.subplots(1 ,figsize=(8, 5))


ax.plot(tide_test['Year'] , tide_test['Sea Level from Tide Gauge Honiara (cm)'], label='lauren')
#ax.plot(tide['Date'], tide['Sea Level'], label='tg')
ax.plot(tide['Date'], (tide['Sea Level']-tide['Sea Level'].mean())/10, label='tide gauge')
ax.plot(tide_hawai['decimal_year'], tide_hawai['ref_corr']/10,'.', label='Hawaii Website')
ax.set_xlim(1994,2012)
ax.set_ylim(-100,100)
ax.legend()

In [ ]:
#tide_hawai.to_excel('hawai_tide_gauge.xlsx')